# GateKeep · notebook 05 · the main comparison: V0 vs V1 vs V2 (all Laya) vs V3 (cascade)

**Attach two inputs:** (1) your `gatekeep_data` dataset (the latest backup folders and the PDF) and (2) **notebook 04's output** (*Add Input → Notebook Outputs*), which holds the fine-tuned models. The restore step takes the cache from the fullest backup and the models from wherever they are.
**Settings:** Accelerator **GPU T4** (one is enough) · Internet **On** · secret `OLLAMA_API_KEY` on.

Run steps 1-3 interactively and check them, then **Save Version → Save & Run All (Commit)**. Expect about **1-1.5 hours** and about **8-10% of the monthly quota**. You are at ~67% before this run; if the usage line passes ~85%, stop and tell Claude.

## 1 · code, packages, GPU, key

In [ ]:
import glob, os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.isdir("GateKeep"):
    subprocess.run(["git", "-C", "GateKeep", "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/RishabhCodezZz/GateKeep.git"], check=True)
os.chdir("GateKeep")
os.environ["PYTHONPATH"] = "src"   # so `!python -m gatekeep.cli` finds the package
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" safetensors huggingface_hub scipy accelerate faiss-cpu sentence-transformers pymupdf pytest openai

In [ ]:
!nvidia-smi -L

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["OLLAMA_API_KEY"] = UserSecretsClient().get_secret("OLLAMA_API_KEY")

## 2 · restore (cache and data from the fullest backup, models from notebook 04's output)

In [ ]:
import sys
sys.path.insert(0, "src")
from gatekeep.restore import restore

COLD_START_OK = False   # set True only if you really want to rebuild everything (costs ~11% of the monthly quota)
where = restore("/kaggle/input", ".")
if where:
    print("restored from", where)
    subprocess.run(["python", "-m", "gatekeep.cli", "purge_empty"], check=True)  # drop empty replies cached before the thinking fix
elif not COLD_START_OK:
    raise SystemExit("STOPPED: no backup found in /kaggle/input. Attach your backup (the dataset must contain the folders "
                     "data/, cache/, results/ or a gatekeep_backup.zip), or set COLD_START_OK = True to rebuild from scratch.")
else:
    print("no backup: cold start, the book is re-parsed and every model call is repeated")
for name in ("review.csv", "handwritten.csv"):
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if found and not os.path.exists(name):
        shutil.copy(found[0], name)
    print(name, "->", "found" if os.path.exists(name) else "MISSING (add it to your dataset)")

In [ ]:
missing = [g for g in ("route", "grade", "grounded", "sufficient") if not os.path.exists(f"models/{g}/model.safetensors")]
assert not missing, (f"fine-tuned weights missing for {missing}. Attach notebook 04's OUTPUT as an input "
                     "(Add Input -> Notebook Outputs); it only has them if it was saved with Save & Run All. Otherwise re-run notebook 04.")
print("fine-tuned models found:", sorted(os.listdir("models")))

## 3 · refresh the per-gate numbers and the cascade thresholds
Route now has three classes (chit-chat was missing before), so Gemma's route numbers and the thresholds must be recomputed. Almost every Gemma call replays from the cache. **Paste back the whole output**, including the `tau` lines.

In [ ]:
!python -m gatekeep.cli evalgates llm,laya-ft

In [ ]:
!cat results/tau.json

## 4 · V2 (all gates by Laya) and V3 (Laya first, Gemma when Laya is unsure) on the held-out-chapter test set
About 343 questions each. **Paste back both result lines and the usage line.**

In [ ]:
!python -m gatekeep.cli variants V2,V3

In [ ]:
!python -m gatekeep.cli breakdown

## 5 · your 80 hand-written questions (the clean test): V0 and V3
V1 on this set is optional and costs about 2% more; run it later only if the quota allows.

In [ ]:
!python -m gatekeep.cli variants V0,V3 "" results/tau.json 0 hw_ data/handwritten.jsonl

## 6 · back up the small files, then stop (the models stay in notebook 04's output)

In [ ]:
!cd /kaggle/working/GateKeep && zip -rq /kaggle/working/gatekeep_backup.zip data cache results models/ckpts.json && ls -lh /kaggle/working/gatekeep_backup.zip

Download `gatekeep_backup.zip` from the **Output** panel and replace the backup in your dataset (new version).